### Linear version of Elastodynamic

### Import of 3D model

In [1]:
import os
os.environ["NGS_NUM_THREADS"] = str(os.cpu_count()-1)

from netgen.stl import STLGeometry
from ngsolve import Mesh as NGSolveMesh


from netgen.occ import *

from ngsolve import *
from ngsolve.webgui import Draw

In [2]:
# Schritt 1: STL-Datei laden


geometry = STLGeometry("Bunny.stl")


# Schritt 2: Mesh aus der STL-Datei erstellen
mesh = geometry.GenerateMesh(maxh=2.5)


# Schritt 3: Mesh in NGSolve importieren
ngsolve_mesh = NGSolveMesh(mesh) 

In [3]:
scene = Draw(ngsolve_mesh, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

In [4]:
E, nu = 210, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)

In [5]:
fes = VectorH1(ngsolve_mesh, order=3, dirichlet="fix")
u,v = fes.TnT()
gfu = GridFunction(fes)

SetNumThreads(12)
with TaskManager():
    print("Defining bilinear form...")
    a = BilinearForm(InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v))).Compile()*dx)
    print("Creating preconditioner...")
    pre = Preconditioner(a, "bddc")
    print("Assembling matrix...")
    a.Assemble()
    print("Matrix Assembly done.")

Defining bilinear form...
Creating preconditioner...
Assembling matrix...
Matrix Assembly done.


In [6]:
force = CF( (0,100000,0) )
f = LinearForm(force*v*ds("force")).Assemble()

In [7]:
from ngsolve.krylovspace import CGSolver
inv = CGSolver(a.mat, pre, printrates=True, tol=1e-8)
gfu.vec.data = inv * f.vec

CG iteration 1, residual = 0.0     


In [8]:
with TaskManager():
    fesstress = MatrixValued(H1(ngsolve_mesh,order=3), symmetric=True)
    gfstress = GridFunction(fesstress)
    gfstress.Interpolate (Stress(Sym(Grad(gfu))))

In [9]:
from ngsolve import *
from ngsolve.webgui import Draw
Draw(gfu, ngsolve_mesh, height="2vh")

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [10]:
Draw (Norm(gfstress), ngsolve_mesh, deformation=1e4*gfu, draw_vol=False, order=3, height="2vh");

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…